# 2.5 文字資料向量化

這份 Notebook 使用 `TextVectorization` 將文字轉成 token index，並建立簡單文字分類模型。

## 1. 載入套件與建立資料

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

tf.keras.utils.set_random_seed(42)
positive = ['this course is useful', 'the model works well', 'great tutorial and clear code', 'tensorflow training is smooth', 'the result is excellent']
negative = ['the model fails', 'bad result and confusing code', 'training is unstable', 'this tutorial is unclear', 'the output is wrong']
texts = np.array((positive * 80) + (negative * 80))
labels = np.array(([1] * len(positive) * 80) + ([0] * len(negative) * 80))
idx = np.random.default_rng(42).permutation(len(texts))
texts, labels = texts[idx], labels[idx]

pd.DataFrame({'text': texts[:5], 'label': labels[:5]})

## 2. 切分資料

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(texts, labels, test_size=0.25, random_state=42, stratify=labels)

x_train_tf = tf.constant(x_train.reshape(-1, 1))
x_test_tf = tf.constant(x_test.reshape(-1, 1))

## 3. 建立 TextVectorization

In [ ]:
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=1000,
    output_mode='int',
    output_sequence_length=8
)
vectorizer.adapt(x_train_tf)

print(vectorizer.get_vocabulary()[:15])
vectorizer(x_train_tf[:3]).numpy()

## 4. 建立文字分類模型

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1,), dtype=tf.string),
    vectorizer,
    tf.keras.layers.Embedding(input_dim=1000, output_dim=16),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(x_train_tf, y_train, validation_split=0.2, epochs=15, batch_size=16, verbose=0)

pd.DataFrame([
    model.evaluate(x_train_tf, y_train, verbose=0, return_dict=True),
    model.evaluate(x_test_tf, y_test, verbose=0, return_dict=True),
], index=['train', 'test'])

## 5. 預測新文字

In [ ]:
new_texts = np.array(['clear tensorflow tutorial', 'the result is wrong and unstable'])
new_texts_tf = tf.constant(new_texts.reshape(-1, 1))
prob = model.predict(new_texts_tf, verbose=0).ravel()
pd.DataFrame({'text': new_texts, 'positive_probability': prob})

## 6. 小結

`TextVectorization` 可直接放進模型，讓文字標準化、切詞與向量化流程與模型一起保存。